Source URL: https://files.consumerfinance.gov/ccdb/complaints.csv.zip
Access date: Jul 13, 2026
File size on disk: 1.41 GB

Prediction: There should be 16 columns based on the codebook.

In [1]:
import pandas as pd
sample = pd.read_csv("../data/raw/complaints.csv.zip", nrows = 100000)

/var/folders/4x/g2gx21dx3jvdqdwv5__x8_gc0000gn/T/ipykernel_51100/2399720025.py:2: DtypeWarning: Columns (0: Consumer complaint narrative) have mixed types. Specify dtype option on import or set low_memory=False.
  sample = pd.read_csv("../data/raw/complaints.csv.zip", nrows = 100000)


In [2]:
print(sample.shape)
print(sample.columns)
print(sample.dtypes)
sample.head()

(100000, 16)
Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='str')
Date received                       str
Product                             str
Sub-product                         str
Issue                               str
Sub-issue                           str
Consumer complaint narrative        str
Company public response             str
Company                             str
State                               str
ZIP code                            str
Tags                                str
Submitted via                       str
Date sent to company                str
Company response to consumer        str
Timely response?                    str
Complaint ID                    float64
dtype: object


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-10-30T01:05:49.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",FL,32960,NaN,Web,2023-10-30T01:16:09.000Z,Closed with non-monetary relief,Yes,7777361.0
1,2023-10-30T01:05:55.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32960,NaN,Web,2023-10-30T01:16:21.000Z,Closed with explanation,Yes,7782528.0
2,2023-10-30T01:05:56.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,32960,NaN,Web,2023-10-30T01:16:33.000Z,Closed with non-monetary relief,Yes,7789854.0
3,2023-11-29T02:27:14.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32960,NaN,Web,2023-11-29T02:37:27.000Z,Closed with explanation,Yes,7914082.0
4,2023-11-29T02:27:06.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",FL,32960,NaN,Web,2023-11-29T02:37:23.000Z,Closed with non-monetary relief,Yes,7913978.0


In [3]:
print(sample['Date received'].min())
print(sample['Date received'].max())

2023-10-30T01:05:49.000Z
2026-07-12T09:07:22.000Z


In [4]:
USE_COLS = [
  'Date received',
  'Product',
  'Sub-product',
  'Issue',
  'Sub-issue',
  # Doesn't include Consumer complaint narrative
  'Company public response',
  'Company',
  'State',
  'ZIP code',
  'Tags',
  'Submitted via',
  'Date sent to company',
  'Company response to consumer',
  'Timely response?',
  'Complaint ID'
]

kept_pieces = []
chunk_size = 500000
rows_scanned = 0
for chunk in pd.read_csv("../data/raw/complaints.csv.zip",
                         chunksize = chunk_size,
                         usecols=USE_COLS,
                         parse_dates=['Date received', 'Date sent to company',]
                         ):
  chunk['Date received'] = pd.to_datetime(chunk['Date received'], utc=True, format='ISO8601', errors='coerce')
  chunk['Date sent to company'] = pd.to_datetime(chunk['Date sent to company'], utc=True, format='ISO8601', errors='coerce')
  rows_scanned += len(chunk)
  mask = chunk['Date received'] >= '2021-01-01'
  kept_pieces.append(chunk[mask])

df = pd.concat(kept_pieces, ignore_index=True)
del kept_pieces
df.to_parquet("../data/processed/complaints_2021_present.parquet")

In [5]:
print(rows_scanned)

16787404


Predictions: 
- `rows_scanned` will total ~5.89 million rows. The customer complaint database currently has about 16.5 million rows, and if the year range of the dataset (found online) is 2012 to 2026, and we are filtering by only 2021-2026, then the proportion of that (assuming an approximately equal amount of complaints per year, which is a BIG assumption), then there will only be about 5.89 million rows.
- This implies that about $\frac{5}{14}$ ths of the rows will survive the filter. 5 years for 2026-2021, and 14 for 2026-2012.


In [6]:
df.shape

(14870477, 15)

In [7]:
df['Date received'].dtype

datetime64[us, UTC]

In [8]:
print(df['Date received'].min())
print(df['Date received'].max())

2021-01-01 00:00:00+00:00
2026-07-12 09:07:22+00:00


In [9]:
df['Date received'].dt.year.value_counts().sort_index()

Date received
2021     495940
2022     800264
2023    1292060
2024    2734283
2025    5443336
2026    4104594
Name: count, dtype: int64

In [10]:
print(df['Complaint ID'][0])
print(df['Complaint ID'][1])
print(df['Complaint ID'][len(df) - 1])

7777361.0
7782528.0
6681348.0


In [ ]:
print(sample['ZIP code'].nunique())
print(sample['State'].nunique())
print(sample['ZIP code'].head(1000).value_counts().head())

10185
57
ZIP code
32960    77
XXXXX    14
55429     8
33033     8
11434     7
Name: count, dtype: int64
